# **(EDL - Edit, transform, Load)**

In [1]:
#import numy 
import numpy as np


In [2]:
#impart pandas
import pandas as pd

## Objectives

* Upload raw data, analyse and transform the date to remove inconsistencies

## Inputs

* The data is onlne retail data in csv form that has been moved this directory dataset\raw\Online_Retail.csv

## Outputs

* The output will be converted dat that has been amedned to remove the following:-
***********************

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [3]:
import os
current_dir = os.getcwd()
current_dir

'c:\\vscode_projects\\Hack1_Online_Retail\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [4]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [5]:
current_dir = os.getcwd()
current_dir

'c:\\vscode_projects\\Hack1_Online_Retail'

# Data extraction

Section 1 content

In [6]:
# set file path
file_path = r'dataset\raw\Online_Retail.csv'

# load csv
df_retail = pd.read_csv(file_path)

#list first 20 rows
df_retail.head(20)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


---

# Check for missleading data and missing data

To get a summary of the data using the describe method.

In [7]:
# copy data set and re-format summary to make it easier to read
summary = df_retail.describe()
summary['Quantity'] = summary['Quantity'].round(0).astype(int)      # integers
summary['CustomerID'] = summary['CustomerID'].round(0).astype(int)  # integers
summary['UnitPrice'] = summary['UnitPrice'].round(2)                # 2 decimals

summary

,Quantity,UnitPrice,CustomerID
count,541909,541909.00,541909
mean,10,4.61,15288
std,218,96.76,1485
min,-80995,-11062.06,12346
25%,1,1.25,14367
50%,3,2.08,15287
75%,10,4.13,16255
max,80995,38970.00,18287


Look for missing data using isnull

In [8]:
missing_data = df_retail[df_retail.isnull().any(axis=1)]
missing_data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,15287,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,15287,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,15287,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,15287,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,15287,United Kingdom
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,15287,United Kingdom
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,15287,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,15287,United Kingdom


In [9]:
# data where the description or customer ID is missing can not be used effectively as the products can not be identified and it cannot be grouped to a customer

count_missing = df_retail[["CustomerID","Description"]].isnull().sum()
count_missing


CustomerID        0
Description    1454
dtype: int64

# Data Analysis

•	The describe function has displayed all data items as floats, there are also too many decimal points to reflect the true value of the data. It is rare to get decimals in quantities for retail, as most retail items are sold in single units. This will require the conversion of the data types.
•	The min is a negative number for quantity; this suggests that returns orders are included in the data set or that there are mistakes in the data. 
•	The max for quantity is extremely high for a retail customer order, this suggests that there are outliers. 
•	The min is a negative number for unit price; this suggests that returns orders are included in the data set or that there are mistakes in the data. 
•	The mean for quality is large for retail customer orders. This suggests that there are outliers in the data.
•	The data with missing descriptions will need to be dropped from the data set


The following action will be taken to clean the data 
1 - Convert quantity to integer
2 - Convert unit price to   float
3 - Convert customer Id to a string
4 - Convert InvoiceDate to date to date and time
5 - Filter out negative quantities for quantity and unit price
6 – Remove missing values in customerID and description
7 - Remove outliers in quantity


In [13]:
#Load the data into a new dataframe for cleaning

df_clean_retail = df_retail.copy()

# 1 Ensure Quantity is integer
df_clean_retail["Quantity"] = df_clean_retail["Quantity"].astype(int)

# 2 Ensure UnitPrice is float (already float, but safe to enforce)
df_clean_retail["UnitPrice"] = df_clean_retail["UnitPrice"].astype(float)

# 3 Convert CustomerID to string (since it's an identifier, not numeric)
df_clean_retail["CustomerID"] = df_clean_retail["CustomerID"].astype("string")

# 4 convert invoice date and time to datetime
df_clean_retail["InvoiceDate"] = pd.to_datetime(df_clean_retail["InvoiceDate"], errors="coerce")

# 5 filter out negative quantities for quantity and unit price
df_clean_retail = df_clean_retail[df_clean_retail["Quantity"] > 0]
df_clean_retail = df_clean_retail[df_clean_retail["UnitPrice"] > 0]

# 6 filter out missing values
df_clean_retail = df_clean_retail.dropna()


# Review Cleaned Data set 

In [16]:
df_clean_retail.head(20)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


In [40]:
# List summary and re-format to make it easier to read
# Exclude InvoiceDate & CustomerID from numeric summary
new_summary = df_clean_retail.drop(columns=['InvoiceDate', 'CustomerID']).describe().round(2)

# Add counts for excluded columns
new_summary.loc['count', 'InvoiceDate'] = df_clean_retail['InvoiceDate'].count()
new_summary.loc['count', 'CustomerID']  = df_clean_retail['CustomerID'].count()

new_summary


,Quantity,UnitPrice,InvoiceDate,CustomerID
count,530104.00,530104.00,530104.0,530104.0
mean,10.54,3.91,NaN,NaN
std,155.52,35.92,NaN,NaN
min,1.00,0.00,NaN,NaN
25%,1.00,1.25,NaN,NaN
50%,3.00,2.08,NaN,NaN
75%,10.00,4.13,NaN,NaN
max,80995.00,13541.33,NaN,NaN


In [41]:
#Check for missing data using isnull
missing_data2 = df_clean_retail[df_clean_retail.isnull().any(axis=1)]
missing_data2

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [45]:
df_clean_retail


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,3,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,3,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3,17850,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,1,12680,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2,12680,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4,12680,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4,12680,France


The count has reduced from 541,909 to 530,104.
The cleansing has removed 11,805 rows of rows containing nulls or negative values.
The cleaned data set now has 0 records with missing data.

# Check for duplicates

The unique dat in the retail 

In [44]:
duplicate_data = df_clean_retail[df_clean_retail.duplicated(subset = ['InvoiceNo','CustomerID', 'StockCode'])]
duplicate_data


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
125,536381,71270,PHOTO CLIP LINE,3,2010-12-01 09:41:00,1,15311,United Kingdom
498,536409,90199C,5 STRAND GLASS NECKLACE CRYSTAL,1,2010-12-01 11:45:00,6,17908,United Kingdom
502,536409,85116,BLACK CANDELABRA T-LIGHT HOLDER,5,2010-12-01 11:45:00,2,17908,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1,17908,United Kingdom
525,536409,90199C,5 STRAND GLASS NECKLACE CRYSTAL,2,2010-12-01 11:45:00,6,17908,United Kingdom
...,...,...,...,...,...,...,...,...
541692,581538,22992,REVOLVER WOODEN RULER,1,2011-12-09 11:34:00,2,14446,United Kingdom
541697,581538,21194,PINK HONEYCOMB PAPER FAN,1,2011-12-09 11:34:00,1,14446,United Kingdom
541698,581538,35004B,SET OF 3 BLACK FLYING DUCKS,1,2011-12-09 11:34:00,5,14446,United Kingdom
541699,581538,22694,WICKER STAR,1,2011-12-09 11:34:00,2,14446,United Kingdom


Add categories to group countries inot regions and to differenciate between producs and non products

In [66]:
#Add country categories using python dictionary

country_region_map = {
    # Europe
    'United Kingdom': 'Europe',
    'France': 'Europe',
    'Germany': 'Europe',
    'Belgium': 'Europe',
    'Netherlands': 'Europe',
    'Spain': 'Europe',
    'Switzerland': 'Europe',
    'Portugal': 'Europe',
    'Italy': 'Europe',
    'Norway': 'Europe',
    'Sweden': 'Europe',
    'Finland': 'Europe',
    'Denmark': 'Europe',
    'Austria': 'Europe',
    'Poland': 'Europe',
    'Lithuania': 'Europe',
    'EIRE': 'Europe',
    'Channel Islands': 'Europe',
    'Cyprus': 'Europe',
    'Greece': 'Europe',
    'Malta': 'Europe',
    'Iceland': 'Europe',
    
    # North America
    'USA': 'North America',
    'Canada': 'North America',
    
    # Asia
    'Japan': 'Asia',
    'Singapore': 'Asia',
    'Hong Kong': 'Asia',
    'Israel': 'Asia',
    
    # Australia / Oceania
    'Australia': 'Oceania',
    
    # Middle East
    'Saudi Arabia': 'Middle East',
    'United Arab Emirates': 'Middle East',

    # Default
    'Unspecified': 'Other'}

#update data set with country categories
df_clean_retail['Region'] = df_clean_retail['Country'].map(country_region_map)
df_clean_retail.head(50)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Region
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Europe
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Europe
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Europe
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Europe
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Europe
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,Europe
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,Europe
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,Europe
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,Europe
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,Europe


In [75]:
#add category for non products and creat separate data set for non product
non_product_terms = ['AMAZONFEE', 'POST', 'S', 'PADS', 'M']

df_clean_retail['ProdCategory'] = df_clean_retail['StockCode'].isin(non_product_terms).map({True: 'Non-Product', False: 'Product'}
                                                                                       )


df_clean_retail.head(50)



,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Region,Category,ProdCategory
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Europe,Product,Product
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Europe,Product,Product
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Europe,Product,Product
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Europe,Product,Product
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Europe,Product,Product
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,Europe,Product,Product
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,Europe,Product,Product
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,Europe,Product,Product
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,Europe,Product,Product
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,Europe,Product,Product


List Non product data

In [77]:
# Summary of data
# Exclude InvoiceDate & CustomerID from numeric summary
cat_summary = df_clean_retail.drop(columns=['InvoiceDate', 'CustomerID']).describe().round(2)

# Add counts for excluded columns
cat_summary.loc['count', 'InvoiceDate'] = df_clean_retail['InvoiceDate'].count()
cat_summary.loc['count', 'CustomerID']  = df_clean_retail['CustomerID'].count()

#new summary by product category
cat_summary_by_prod = df_clean_retail.groupby('ProdCategory').describe().round(2)

#new summary by country
cat_summary_by_country = df_clean_retail.groupby('Country').describe().round(2)

#new summary by region
cat_summary_by_region = df_clean_retail.groupby('Region').describe().round(2)





In [78]:
cat_summary


,Quantity,UnitPrice,InvoiceDate,CustomerID
count,530104.00,530104.00,530104.0,530104.0
mean,10.54,3.91,NaN,NaN
std,155.52,35.92,NaN,NaN
min,1.00,0.00,NaN,NaN
25%,1.00,1.25,NaN,NaN
50%,3.00,2.08,NaN,NaN
75%,10.00,4.13,NaN,NaN
max,80995.00,13541.33,NaN,NaN


In [80]:
cat_summary_by_prod 


Quantity                                              \
                 count   mean  min  25%  50%   75%      max    std   
ProdCategory                                                         
Non-Product     1454.0   7.14  1.0  1.0  2.0   4.0   1600.0   62.4   
Product       528650.0  10.55  1.0  1.0  3.0  11.0  80995.0  155.7   

             InvoiceDate                                 ...  \
                   count                           mean  ...   
ProdCategory                                             ...   
Non-Product         1454  2011-07-04 15:40:48.528198144  ...   
Product           528650  2011-07-04 20:16:50.652644096  ...   

                                       UnitPrice                            \
                              max  std     count   mean  min    25%    50%   
ProdCategory                                                                 
Non-Product   2011-12-09 12:16:00  NaN    1454.0  84.54  0.0  15.00  18.00   
Product       2011-12-09 12:50:00  NaN  528650.0   3.69  0.0   1.25   2.08   

                                       
                75%       max     std  
ProdCategory                           
Non-Product   18.00  13541.33  520.43  
Product        4.13  11062.06   23.05  

[2 rows x 24 columns]

In [81]:
cat_summary_by_country 

Quantity                                           \
                         count   mean  min    25%   50%    75%      max   
Country                                                                   
Australia               1182.0  70.98  1.0  10.00  32.0  100.0   1152.0   
Austria                  398.0  12.26  1.0   6.00  10.0   12.0    288.0   
Bahrain                   18.0  17.44  2.0   6.00   6.0   11.0     96.0   
Belgium                 2031.0  11.44  1.0   4.00  10.0   12.0    272.0   
Brazil                    32.0  11.12  2.0   3.00  10.0   18.0     24.0   
Canada                   151.0  18.30  1.0   6.00  12.0   20.0    504.0   
Channel Islands          748.0  12.69  1.0   4.00  10.0   12.0    407.0   
Cyprus                   614.0  10.36  1.0   2.00   5.0   12.0    288.0   
Czech Republic            25.0  26.84  1.0  18.00  24.0   24.0     72.0   
Denmark                  380.0  21.67  1.0  12.00  12.0   24.0    256.0   
EIRE                    7890.0  18.65  1.0   4.00  12.0   12.0   1440.0   
European Community        60.0   8.32  1.0   3.75   6.0   12.0     24.0   
Finland                  685.0  15.63  1.0   6.00  10.0   16.0    144.0   
France                  8407.0  13.33  1.0   6.00  10.0   12.0    912.0   
Germany                 9040.0  13.19  1.0   6.00  10.0   12.0    600.0   
Greece                   145.0  10.74  1.0   6.00  10.0   12.0     48.0   
Hong Kong                284.0  16.81  1.0   6.00  12.0   24.0    144.0   
Iceland                  182.0  13.51  2.0   6.00  12.0   12.0    240.0   
Israel                   295.0  14.95  1.0   4.00  12.0   24.0    100.0   
Italy                    758.0  10.70  1.0   4.00   8.0   12.0    200.0   
Japan                    321.0  81.05  1.0  12.00  48.0   72.0   2040.0   
Lebanon                   45.0   8.58  2.0   6.00   8.0   12.0     24.0   
Lithuania                 35.0  18.63  6.0  12.00  16.0   24.0     48.0   
Malta                    112.0   8.66  1.0   4.00   6.0   12.0     48.0   
Netherlands             2359.0  84.93  1.0  16.00  72.0  100.0   2400.0   
Norway                  1071.0  18.05  1.0   6.00  12.0   24.0    240.0   
Poland                   330.0  11.16  1.0   4.00  10.0   12.0     72.0   
Portugal                1501.0  10.83  1.0   4.00   9.0   12.0    120.0   
RSA                       57.0   6.16  2.0   3.00   6.0   10.0     12.0   
Saudi Arabia               9.0   8.89  2.0   6.00  12.0   12.0     12.0   
Singapore                222.0  23.61  1.0   8.50  12.0   24.0    288.0   
Spain                   2484.0  11.25  1.0   3.00   6.0   12.0    360.0   
Sweden                   451.0  80.01  1.0   9.00  24.0   96.0    768.0   
Switzerland             1966.0  15.58  1.0   6.00  12.0   20.0    288.0   
USA                      179.0  13.73  1.0   6.00  12.0   16.0     72.0   
United Arab Emirates      68.0  14.44  1.0   6.00  12.0   12.0     72.0   
United Kingdom        485123.0   9.61  1.0   1.00   3.0   10.0  80995.0   
Unspecified              446.0   7.40  1.0   1.00   3.0   12.0     48.0   

                             InvoiceDate                                 ...  \
                         std       count                           mean  ...   
Country                                                                  ...   
Australia              98.76        1182  2011-06-06 22:15:06.497462016  ...   
Austria                21.59         398  2011-07-29 07:22:26.231155968  ...   
Bahrain                25.88          18  2011-05-04 01:12:36.666666496  ...   
Belgium                13.60        2031  2011-07-11 20:24:21.654357504  ...   
Brazil                  8.48          32            2011-04-15 10:25:00  ...   
Canada                 46.68         151  2011-06-26 16:27:03.576159232  ...   
Channel Islands        22.67         748  2011-06-30 00:30:55.508021504  ...   
Cyprus                 23.32         614  2011-06-04 06:29:50.130293248  ...   
Czech Republic         17.28          25            2011-05-27 19:

In [82]:
cat_summary_by_region

Quantity                                                  \
                  count   mean  min   25%   50%    75%      max     std   
Region                                                                    
Asia             1122.0  36.04  1.0   6.0  12.0   36.0   2040.0  102.29   
Europe         526710.0  10.35  1.0   1.0   3.0   10.0  80995.0  155.85   
Middle East        77.0  13.79  1.0   6.0  12.0   12.0     72.0   11.92   
North America     330.0  15.82  1.0   6.0  12.0   19.5    504.0   32.71   
Oceania          1182.0  70.98  1.0  10.0  32.0  100.0   1152.0   98.76   
Other             446.0   7.40  1.0   1.0   3.0   12.0     48.0    8.77   

              InvoiceDate                                 ...  \
                    count                           mean  ...   
Region                                                    ...   
Asia                 1122  2011-05-31 22:57:38.930481152  ...   
Europe             526710  2011-07-04 22:46:39.141994496  ...   
Middle East            77  2011-05-25 06:29:29.610389248  ...   
North America         330            2011-08-17 16:08:26  ...   
Oceania              1182  2011-06-06 22:15:06.497462016  ...   
Other                 446  2011-07-30 15:13:21.659192832  ...   

                                        UnitPrice                           \
                               max  std     count   mean   min   25%   50%   
Region                                                                       
Asia           2011-11-29 15:52:00  NaN    1122.0  19.02  0.06  0.97  1.85   
Europe         2011-12-09 12:50:00  NaN  526710.0   3.88  0.00  1.25  2.08   
Middle East    2011-09-22 13:00:00  NaN      77.0   3.26  0.29  1.25  1.65   
North America  2011-12-05 10:14:00  NaN     330.0   4.01  0.10  0.85  1.65   
Oceania        2011-11-24 12:30:00  NaN    1182.0   3.06  0.19  1.25  1.79   
Other          2011-11-24 14:55:00  NaN     446.0   2.70  0.19  0.85  1.65   

                                       
                75%       max     std  
Region                                 
Asia           3.75   3949.32  200.66  
Europe         4.13  13541.33   34.80  
Middle East    2.95     37.50    5.05  
North America  2.95    550.94   30.30  
Oceania        3.75    350.00   10.39  
Other          3.35     16.95    2.87  

[6 rows x 24 columns]

# Remove Outliers

Identify range of Outliers and use requirements from the customer ot set max ranges for quantity.  The customer would like amazon fees, non product fees and postage pees to be examined separately to the products.



In [83]:
#Stores outliers in data set - These have been identified as where the quantities are  greater than 3200 and unit price is less than 700
outliers = df_clean_retail[
    (df_clean_retail['Quantity'] <= 0) | (df_clean_retail['Quantity'] > 3200) |
    (df_clean_retail['UnitPrice'] <= 0) | (df_clean_retail['UnitPrice'] > 700)
]
# removes outliers from data set for retail product analysis
df_no_outliers = df_clean_retail.drop(outliers.index)

# remove data where country not specified
df_no_outliers = df_no_outliers[df_no_outliers['Country'] != 'Unspecified']

outliers


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Region,Category,ProdCategory
6165,536876,DOT,DOTCOM POSTAGE,1,2010-12-03 11:36:00,887.52,15287,United Kingdom,Europe,Product,Product
10812,537237,DOT,DOTCOM POSTAGE,1,2010-12-06 09:58:00,863.74,15287,United Kingdom,Europe,Product,Product
11381,537240,DOT,DOTCOM POSTAGE,1,2010-12-06 10:08:00,940.87,15287,United Kingdom,Europe,Product,Product
13924,537434,DOT,DOTCOM POSTAGE,1,2010-12-06 16:57:00,950.99,15287,United Kingdom,Europe,Product,Product
14392,537534,M,Manual,1,2010-12-07 11:48:00,924.59,15287,United Kingdom,Europe,Non-Product,Non-Product
...,...,...,...,...,...,...,...,...,...,...,...
537254,581238,DOT,DOTCOM POSTAGE,1,2011-12-08 10:53:00,1683.75,15287,United Kingdom,Europe,Product,Product
539368,581439,DOT,DOTCOM POSTAGE,1,2011-12-08 16:30:00,938.59,15287,United Kingdom,Europe,Product,Product
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,Europe,Product,Product
540908,581492,DOT,DOTCOM POSTAGE,1,2011-12-09 10:03:00,933.17,15287,United Kingdom,Europe,Product,Product


Summary of data

In [84]:
# List summary and re-format to make it easier to read, separate out non numeric and create counts
clean_summary = df_no_outliers.drop(columns=['InvoiceDate', 'CustomerID']).describe().round(2)

# Add counts for excluded columns
clean_summary.loc['count', 'InvoiceDate'] = df_no_outliers['InvoiceDate'].count()
clean_summary.loc['count', 'CustomerID']  = df_no_outliers['CustomerID'].count()

clean_summary


,Quantity,UnitPrice,InvoiceDate,CustomerID
count,529556.00,529556.00,529556.0,529556.0
mean,10.23,3.60,NaN,NaN
std,36.37,10.28,NaN,NaN
min,1.00,0.00,NaN,NaN
25%,1.00,1.25,NaN,NaN
50%,3.00,2.08,NaN,NaN
75%,10.00,4.13,NaN,NaN
max,3186.00,700.00,NaN,NaN


# Conclusions and next steps

This data set called df_no_outliers can now be used for analysis of product sales.  

The data set df_clean_retail can be used to investigate non-product cost against total product cost in a future project.

The data set called no-outliers will now be writtent to a data set that will be used by the data visualisations jupiter notebook.

In [85]:
#save data sets to csv files
df_clean_retail.to_csv('dataset/processed/clean_retail_data.csv', index=False)
df_no_outliers.to_csv('dataset/processed/no_outliers_retail_data.csv', index=False)